## Synthetic data for ASQA

Since HotpotQA provides short-form answers, we apply an answer verbalization step to transform entity-based answers into complete natural language sentences, ensuring compatibility with LLM-based evaluation metrics such as RAGAs.

In [22]:
import pandas as pd
import numpy as np
import os
import dotenv
from typing import Any, Dict, List, Optional
import time
import ast
from google import genai
import google.genai
from google.genai import types
from openai import OpenAI
import asyncio
from asyncio import Semaphore
from tqdm.auto import tqdm  # Auto-select best version (notebook or console)
from dotenv import load_dotenv

load_dotenv()

True

## 1. Configuration and Setup

In [35]:
ORIGINAL_DATA_PATH = "../data/asqa/train.csv"
HALLU_PATH = "../data/asqa/hallu_asqa.csv"
LABELED_PATH = "../data/asqa/labeled_asqa.csv"
CHECKPOINT_PATH = "../results/synthetic-data/asqa_checkpoint.csv"

In [4]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# client = genai.Client(api_key=GOOGLE_API_KEY)
client = OpenAI(api_key=OPENAI_API_KEY)
SLEEP_TIME = 1

In [7]:
for m in client.models.list():
    print(m.id)

text-embedding-ada-002
whisper-1
gpt-3.5-turbo
tts-1
gpt-3.5-turbo-16k
gpt-4-0613
gpt-4
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o-audio-preview
gpt-4o-realtime-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
o1-2024-12-17
o1
gpt-4o-mini-realtime-preview
gpt-4o-mini-audio-preview
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03-19
o1-pro
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2

## 2. Data Loading and Preprocessing

In [11]:
# load dataset
asqa = pd.read_csv(ORIGINAL_DATA_PATH)

In [6]:
asqa.head()

,id,question,answer,context,supporting_facts
0,asqa_0,When does the new bunk'd come out?,The new bunk'd episode 41 comes out on April 2...,"{'title': [""List of Bunk'd episodes"", 'QA_1', ...","{'title': [""List of Bunk'd episodes""]}"
1,asqa_1,Who won the 2016 ncaa football national champi...,The 2015 - 2016 season's ncaa national footbal...,{'title': ['2015 College Football Playoff Nati...,{'title': ['2016 College Football Playoff Nati...
2,asqa_2,When was the last time the death penalty was u...,The last time the death penalty was used in pa...,{'title': ['Capital punishment in Pennsylvania...,{'title': ['Gary M. Heidnik']}
3,asqa_3,Where will failure of the left ventricle cause...,"""Backward"" failure of the left ventricle cause...","{'title': ['Heart failure', 'QA_1', 'QA_2'], '...",{'title': ['Heart failure']}
4,asqa_4,Who won the war between ethiopia and italy?,The first war between Italy and Ethiopia took ...,"{'title': ['Second Italo-Ethiopian War', 'Firs...","{'title': ['First Italo-Ethiopian War', 'Secon..."


In [7]:
asqa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4353 entries, 0 to 4352
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                4353 non-null   object
 1   question          4353 non-null   object
 2   answer            4353 non-null   object
 3   context           4353 non-null   object
 4   supporting_facts  4353 non-null   object
dtypes: object(5)
memory usage: 170.2+ KB


In [5]:
print(asqa.loc[[1]])


       id                                           question  \
1  asqa_1  Who won the 2016 ncaa football national champi...   

                                              answer  \
1  The 2015 - 2016 season's ncaa national footbal...   

                                             context  \
1  {'title': ['2015 College Football Playoff Nati...   

                                    supporting_facts  
1  {'title': ['2016 College Football Playoff Nati...  


In [8]:
def parse_supporting_facts(supporting_facts_raw, context_raw):
    try:
        # ===== clean string =====
        sf_clean = supporting_facts_raw.replace('""', '"')
        ctx_clean = context_raw.replace('""', '"')

        sf = ast.literal_eval(sf_clean)
        ctx = ast.literal_eval(ctx_clean)

        # ===== lấy danh sách title cần dùng =====
        sf_titles = set(sf.get("title", []))

        ctx_titles = ctx.get("title", [])
        ctx_sentences = ctx.get("sentences", [])

        facts = []

        # ===== lọc đúng doc theo title =====
        for title, sents in zip(ctx_titles, ctx_sentences):
            if title in sf_titles or str(title).startswith("QA"):
                joined_sents = " ".join(sents)
                
                facts.append(f"- ({title}) {joined_sents}")

        return "\n".join(facts)

    except Exception as e:
        return ""

In [12]:
asqa['context'] = asqa.apply(lambda row: parse_supporting_facts(row['supporting_facts'], row['context']), axis=1)
display(asqa.head(3))

,id,question,answer,context,supporting_facts
0,asqa_0,When does the new bunk'd come out?,The new bunk'd episode 41 comes out on April 2...,- (List of Bunk'd episodes) The new bunk'd epi...,"{'title': [""List of Bunk'd episodes""]}"
1,asqa_1,Who won the 2016 ncaa football national champi...,The 2015 - 2016 season's ncaa national footbal...,- (2016 College Football Playoff National Cham...,{'title': ['2016 College Football Playoff Nati...
2,asqa_2,When was the last time the death penalty was u...,The last time the death penalty was used in pa...,"- (QA_1) As of 2017, when was the last time th...",{'title': ['Gary M. Heidnik']}


## 3. Prompt Design and LLM Interaction

In [13]:
PROMPT_TEMPLATE = """
You are given a question, a correct answer, and supporting facts extracted from reliable sources.

Your task is to generate a hallucinated answer.

=====================
Requirements:
- The answer MUST be factually incorrect based on the supporting facts
- The answer MUST directly conflict with at least one key fact (e.g., date, number, entity)
- The answer should remain fluent, natural, and confident
- Keep the style and length similar to the original answer (1–2 sentences)
- DO NOT mention uncertainty or say it is incorrect
- DO NOT use any special formatting (e.g., bold, italics, bullet points, markdown, or symbols)
- Output plain text only

=====================
Hallucination Strategy:
- Modify ONLY 1–2 key facts from the supporting facts
- Keep all other details consistent with the original answer
- Prefer subtle contradictions over obvious errors

=====================
Input:

Question:
{question}

Correct Answer:
{correct_answer}

Context:
{context}

=====================
Output:
Return ONLY the hallucinated answer.
"""

PROMPT_SYSTEM = """You are an AI that generates hallucinated answers for evaluation purposes.

Follow all instructions strictly:
- The answer MUST be factually incorrect based on the supporting facts
- The answer MUST directly conflict with at least one key fact (e.g., date, number, entity)
- The answer should remain fluent, natural, and confident
- Keep the style and length similar to the original answer (1–2 sentences)
- DO NOT mention uncertainty or say it is incorrect
- DO NOT use any special formatting
- Output plain text only

Hallucination Strategy:
- Modify ONLY 1–2 key facts
- Keep all other details consistent
- Prefer subtle contradictions over obvious errors
"""

PROMPT_USER = """Question:
{question}

Correct Answer:
{correct_answer}

Context:
{context}

Return ONLY the hallucinated answer.
"""

In [14]:
def synthetic(row, client, model="gpt-4o-mini"):
    
    prompt_user = PROMPT_USER.format(
        question=row["question"].strip(),
        correct_answer=row["answer"].strip(),
        context=row["context"].strip()
    )

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": PROMPT_SYSTEM},
                {"role": "user", "content": prompt_user}
            ],
            temperature=0.7
        )
        text = response.choices[0].message.content
        # basic validation
        if not text or text == row["answer"]:
           return None
        
        return text
    
    except Exception:
        return None



## 4. Test on 1 sample

In [15]:
row = asqa.iloc[3]  # lấy sample đầu tiên

hallu = synthetic(
    row=row,
    client=client,
    model="gpt-4o-mini"
)

print("=== QUESTION ===")
print(row["question"])

print("\n=== ORIGINAL ANSWER ===")
print(row["answer"])

print("\n=== HALLUCINATED ANSWER ===")
print(hallu)

=== QUESTION ===
Where will failure of the left ventricle cause increased pressure?

=== ORIGINAL ANSWER ===
"Backward" failure of the left ventricle causes congestion of the lungs' blood vessels, and therefore causes increased pressure in the lungs. These symptoms are predominantly respiratory in nature.

=== HALLUCINATED ANSWER ===
"Backward" failure of the left ventricle causes congestion of the liver's blood vessels, and therefore causes increased pressure in the liver. These symptoms are predominantly digestive in nature.


## 5. Batch generation with checkpoint resume

In [16]:
asqa = asqa.reset_index(drop=True)

In [26]:
def load_checkpoint(checkpoint_path):
    if checkpoint_path and os.path.exists(checkpoint_path):
        return pd.read_csv(checkpoint_path, dtype={"id": str})
    return pd.DataFrame(columns=["id", "hallu_answer"])

def save_checkpoint(rows: List[Dict[str, str]], checkpoint_path: str) -> None:
    ckpt = pd.DataFrame(rows)
    ckpt.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

def synthetic_batch(df, client, model="gpt-4o-mini",
                    checkpoint_path=None, checkpoint_freq=100):
    
    df = df.copy()
    df["id"] = df["id"].astype(str)

    checkpoint_df = load_checkpoint(checkpoint_path) if checkpoint_path else pd.DataFrame(columns=["id", "hallu_answer"])
    done_ids = set(checkpoint_df["id"].astype(str).tolist())

    print(f"🔄 Resuming from checkpoint: {len(done_ids)} samples already processed.")

    rows = checkpoint_df.to_dict("records")

    pbar = tqdm(total=len(df), initial=len(done_ids), desc="Generating hallucinations", unit="sample")


    for _, row in df.iterrows():
        sample_id = str(row["id"])

        if sample_id in done_ids:
            continue

        hallucinated = synthetic(row=row, client=client, model=model)
        if hallucinated is None:
            hallucinated = ""

        rows.append({
            "id": sample_id,
            "hallu_answer": hallucinated,
        })
        done_ids.add(sample_id)

        pbar.update(1)

        if checkpoint_path and len(rows) % checkpoint_freq == 0:
            save_checkpoint(rows, checkpoint_path)
            print(f"💾 checkpoint saved: {len(rows)}/{len(df)}")

        time.sleep(SLEEP_TIME)

    pbar.close()

    if checkpoint_path:
        save_checkpoint(rows, checkpoint_path)
        print(f"✅ final checkpoint saved: {checkpoint_path}")

    return pd.DataFrame(rows)

In [32]:
def build_hallu_dataset(
    base_df: pd.DataFrame,
    client: OpenAI,
    model: str = "gpt-4o-mini",
    checkpoint_path: str = CHECKPOINT_PATH,
    hallu_path: str = HALLU_PATH,
    labeled_path: str = LABELED_PATH,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    base_df = base_df.copy()
    base_df["id"] = base_df["id"].astype(str)

    # 1. Generate hallucinated answers
    ckpt = synthetic_batch(
        base_df,
        client=client,
        model=model,
        checkpoint_path=checkpoint_path,
        checkpoint_freq=25,
    )

    ckpt["id"] = ckpt["id"].astype(str)

    # 2. Merge back with original samples
    hallu_df = base_df.merge(ckpt, on="id", how="left")

    # 3. Filter empty generations
    empty_mask = hallu_df["hallu_answer"].fillna("").str.len() < 5
    n_empty = int(empty_mask.sum())
    print(f"Empty/invalid hallucinations: {n_empty}")

    if n_empty > 0:
        print("Retrying empty/invalid generations once...")
        retry_df = hallu_df.loc[empty_mask, base_df.columns].copy()
        retry_rows = []

        for _, row in tqdm(retry_df.iterrows(), total=len(retry_df), desc="Retrying"):
            hallucinated = synthetic(row=row, client=client, model=model)
            retry_rows.append({
                "id": str(row["id"]),
                "hallu_answer": hallucinated or "",
            })
            time.sleep(SLEEP_TIME)

        retry_ckpt = pd.DataFrame(retry_rows)
        retry_map = dict(zip(retry_ckpt["id"].astype(str), retry_ckpt["hallu_answer"]))

        hallu_df.loc[empty_mask, "hallu_answer"] = hallu_df.loc[empty_mask, "id"].map(retry_map)

    hallu_df = hallu_df[
        hallu_df["hallu_answer"].notnull() &
        (hallu_df["hallu_answer"].str.len() >= 5)
    ].copy()

    # 4. Export hallu dataset with same schema as gold dataset + answer field
    hallu_export = hallu_df[["id", "question", "context", "answer", "hallu_answer"]].copy()
    hallu_export.to_csv(hallu_path, index=False, encoding="utf-8-sig")
    print(f"Saved hallucinated dataset: {hallu_path}")

    # 5. Build labeled dataset
    # label = 1: supported/gold answer
    # label = 0: hallucinated/unsupported answer
    positive = base_df[["id", "question", "context", "answer"]].copy()
    positive["label"] = 1

    negative = hallu_export[["id", "question", "context", "hallu_answer"]].copy()
    negative = negative.rename(columns={"hallu_answer": "answer"})
    negative["id"] = negative["id"].astype(str) + "_hallu"
    negative["label"] = 0

    labeled = pd.concat([positive, negative], ignore_index=True)
    labeled.to_csv(labeled_path, index=False, encoding="utf-8-sig")
    print(f"Saved labeled dataset: {labeled_path}")
    print(labeled["label"].value_counts())

    return labeled, hallu_export


In [33]:
asqa_final, asqa_hallu = build_hallu_dataset(
    base_df=asqa,
    client=client,
    model="gpt-4o-mini",
    checkpoint_path=CHECKPOINT_PATH,
    hallu_path=HALLU_PATH,
    labeled_path=LABELED_PATH
)

🔄 Resuming from checkpoint: 4353 samples already processed.


Generating hallucinations: 100%|██████████| 4353/4353 [00:00<?, ?sample/s]


✅ final checkpoint saved: ../data/asqa/checkpoint.csv
Empty/invalid hallucinations: 0
Saved hallucinated dataset: ../data/asqa/hallu_asqa.csv
Saved labeled dataset: ../data/asqa/labeled_asqa.csv
label
1    4353
0    4353
Name: count, dtype: int64


## 6. Final dataset overview

In [34]:
print(f"Save hallucination dataset to {HALLU_PATH}")
print(asqa_hallu.shape)
display(asqa_hallu.head(3))

print(f"Save final dataset to {LABELED_PATH}")
print(asqa_final.shape)
display(asqa_final.head(3))

Save hallucination dataset to ../data/asqa/hallu_asqa.csv
(4353, 5)


,id,question,context,answer,hallu_answer
0,asqa_0,When does the new bunk'd come out?,- (List of Bunk'd episodes) The new bunk'd epi...,The new bunk'd episode 41 comes out on April 2...,The new bunk'd episode 41 comes out on April 2...
1,asqa_1,Who won the 2016 ncaa football national champi...,- (2016 College Football Playoff National Cham...,The 2015 - 2016 season's ncaa national footbal...,The 2015 - 2016 season's NCAA national footbal...
2,asqa_2,When was the last time the death penalty was u...,"- (QA_1) As of 2017, when was the last time th...",The last time the death penalty was used in pa...,The last time the death penalty was used in PA...


Save final dataset to ../data/asqa/labeled_asqa.csv
(8706, 5)


,id,question,context,answer,label
0,asqa_0,When does the new bunk'd come out?,- (List of Bunk'd episodes) The new bunk'd epi...,The new bunk'd episode 41 comes out on April 2...,1
1,asqa_1,Who won the 2016 ncaa football national champi...,- (2016 College Football Playoff National Cham...,The 2015 - 2016 season's ncaa national footbal...,1
2,asqa_2,When was the last time the death penalty was u...,"- (QA_1) As of 2017, when was the last time th...",The last time the death penalty was used in pa...,1
